## Bitcoin Forensics: 
**Detecting Ransomware Addresses with a Perceptron Network**

### 1. Problem to Solve

Bitcoin transactions are public, but identifying addresses connected to ransomware activity is still difficult. This project uses the BitcoinHeist Ransomware Dataset to classify Bitcoin addresses as either normal or ransomware-related.

The goal is to train a perceptron-based model that detects suspicious addresses from transaction graph features such as chain length, number of neighbors, transaction count, looped transactions, weight, and income.

### 2. Setup
Imports all the necessary dependencies for the project

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
from sklearn.dummy import DummyClassifier
from sklearn.metrics import balanced_accuracy_score

print("Environment check")
print("-----------------")
print(f"Python version: {sys.version.split()[0]}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

data_path = project_root / "data" / "BitcoinHeistData.csv"
figures_path = project_root / "outputs" / "figures"

print(f"Project root: {project_root}")
print(f"Data path exists: {data_path.exists()}")
print(f"Figures path exists: {figures_path.exists()}")

Environment check
-----------------
Python version: 3.13.5
NumPy version: 2.5.2
Pandas version: 3.0.5
Project root: /Users/victorvulturescu/Documents/RustWorks/IpWorkshop/bitcoin-forensics
Data path exists: True
Figures path exists: True


## 3. Load the dataset

This section loads the BitcoinHeist dataset into a pandas DataFrame so it can be inspected, cleaned, and prepared for machine learning.

In [3]:
data_path = project_root / "data" / "BitcoinHeistData.csv"

USE_SAMPLE = False
SAMPLE_ROWS = 200_000

if USE_SAMPLE:
    df = pd.read_csv(data_path, nrows=SAMPLE_ROWS)
else:
    df = pd.read_csv(data_path)

df = pd.read_csv(data_path)

print(f"Dataset loaded successfully.")
print(f"Shape: {df.shape[0]:,} rows and {df.shape[1]} columns")

df.head()

Dataset loaded successfully.
Shape: 2,916,697 rows and 10 columns


,address,year,day,length,weight,count,looped,neighbors,income,label
0,111K8kZAEnJg245r2cM6y9zgJGHZtJPy6,2017,11,18,0.008333,1,0,2,100050000.0,princetonCerber
1,1123pJv8jzeFQaCV4w644pzQJzVWay2zcA,2016,132,44,0.000244,1,0,1,100000000.0,princetonLocky
2,112536im7hy6wtKbpH1qYDWtTyMRAcA2p7,2016,246,0,1.000000,1,0,2,200000000.0,princetonCerber
3,1126eDRw2wqSkWosjTCre8cjjQW8sSeWH7,2016,322,72,0.003906,1,0,2,71200000.0,princetonCerber
4,1129TSjKtx65E35GiUo4AYVeyo48twbrGX,2016,238,144,0.072848,456,0,1,200000000.0,princetonLocky


# 4.Understanding the dataset
This section explores the structure of the dataset, the meaning of each column, and the distribution of `white` versus ransomware-related Bitcoin addresses.
The meaning of each of the collume is broken down as followes

| Column | Meaning | How we use it |
|---|---|---|
| `address` | Bitcoin address identifier | Removed before training, as its an ID, not a behaviour feature |
|`year`|Year of the transaction|Feature, may reflect time trends|
|`day`|Day of the year in which the transation was made|Feature, may reflect time trends|
|`length`|Length of the transaction chain connected to the address|Feature|
|`weight`|A graph-based weight measuring transaction flow/splitting behavio|Feature|
|`count`|Number of transactions or reachable outputs in the chain|Feature|
|`looped`|Amount/count of transactions that loop back in the graph|Feature|
|`neighbors`|Number of neighboring addresses connected in the graph|Feature|
|`income`|Amount received by the address, in satoshis|Feature|
|`label`|`white` for clean transactions, ransomware family name otherwise|Target Variable|

In [4]:
print(f"Rows: {df.shape[0]:,}")
print(f"Collumns: {df.shape[1]}")

df.head()

Rows: 2,916,697
Collumns: 10


,address,year,day,length,weight,count,looped,neighbors,income,label
0,111K8kZAEnJg245r2cM6y9zgJGHZtJPy6,2017,11,18,0.008333,1,0,2,100050000.0,princetonCerber
1,1123pJv8jzeFQaCV4w644pzQJzVWay2zcA,2016,132,44,0.000244,1,0,1,100000000.0,princetonLocky
2,112536im7hy6wtKbpH1qYDWtTyMRAcA2p7,2016,246,0,1.000000,1,0,2,200000000.0,princetonCerber
3,1126eDRw2wqSkWosjTCre8cjjQW8sSeWH7,2016,322,72,0.003906,1,0,2,71200000.0,princetonCerber
4,1129TSjKtx65E35GiUo4AYVeyo48twbrGX,2016,238,144,0.072848,456,0,1,200000000.0,princetonLocky


## 5. Preparing the Data
This section converts the dataset in to a binary classification problem, removes problems that are not necesarily helpfull for the problem that we want to solve and separates the input feratures from the target

In [5]:

# First we will create a coppy so the original dataframe stays intact
df_prepared = df.copy()

# Than we will convert the original labels in to binary form
# For the label column, we do not care about the particular ransomware family here. White addresses become 0, and ransomware-related addresses become 1.
df_prepared["is_ransomware"] = (df_prepared["label"] != "white").astype(int)

# For the address is an identifier, not a feature column so we can safely drop it
feature_columns = [
    "year",
    "day",
    "length",
    "weight",
    "count",
    "looped",
    "neighbors",
    "income",
]

# Now we will remove rows with missing feature or target values, if any exist
df_prepared[feature_columns + ["is_ransomware"]].isna().sum()

# The features
X = df_prepared[feature_columns]
# The target
y = df_prepared["is_ransomware"]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

df_prepared[["label", "is_ransomware"]].head()


Features shape: (2916697, 8)
Target shape: (2916697,)


,label,is_ransomware
0,princetonCerber,1
1,princetonLocky,1
2,princetonCerber,1
3,princetonCerber,1
4,princetonLocky,1


In [6]:
# Remove rows with missing feature or target values, if any exist
df_prepared = df_prepared.dropna(subset=feature_columns + ["is_ransomware"])

# The features
X = df_prepared[feature_columns]

# The target
y = df_prepared["is_ransomware"]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

Features shape: (2916697, 8)
Target shape: (2916697,)


Now we can inspect the prepared dataset and check the percentage of normal and ransomware-related addresses.

In [7]:
class_counts = df_prepared["is_ransomware"].value_counts().sort_index()
class_percentages = df_prepared["is_ransomware"].value_counts(normalize=True).sort_index() * 100

class_summary = pd.DataFrame({
    "class": ["normal", "ransomware"],
    "count": class_counts.values,
    "percentage": class_percentages.round(2).values,
})

class_summary

,class,count,percentage
0,normal,2875284,98.58
1,ransomware,41413,1.42


## 6.Train Test Split
This section is dedicated to spliting the data set in to a a set dedicatde to training and a set dedicated to testing the model. We will use a stratified split so that both sets keep the same proportionality of ransomware-related to white transactions.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, # Contains the input features
    y, # Contains the target that we want to predict
    test_size=0.2, # 20 % of the data is keped asside for testing
    random_state=42, #the value makes the split reproductibile, the randomnes is to not have the same exact train / split each time
    stratify=y # Keeps the class ratio similar in both training and testing sets. This is important because the dataset has many more normal addresses than ransomware addresses.
    
)

print(f"Training set: {X_train.shape[0]:,} rows")
print(f"Testing set: {X_test.shape[0]:,} rows")

print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nTesting class distribution:")
print(y_test.value_counts(normalize=True).sort_index())

Training set: 2,333,357 rows
Testing set: 583,340 rows

Training class distribution:
is_ransomware
0    0.985802
1    0.014198
Name: proportion, dtype: float64

Testing class distribution:
is_ransomware
0    0.985801
1    0.014199
Name: proportion, dtype: float64


## 7. Balancing and Augmening the Data
Durning the `Prepairing the Data` section, we can clearly notice that the dataset is extreamly imbalanced. The normal transactions hevily outway the ransomware ones. This imbalance will have a huge impact on the pattern recignition of our model, it will only be capabile of predicting the majority class. So for the training data, we must undersample the `white` adresses to better balance the moodel. The testing data however must remain untuched, because the model must be tested on **real** , **unchanged** data. 

In [9]:
# Separate the original training data by class
train_df = X_train.copy()
train_df["is_ransomware"] = y_train.values

normal_train = train_df[train_df["is_ransomware"] == 0]
ransomware_train = train_df[train_df["is_ransomware"] == 1]

print("Original training class distribution:")
print(train_df["is_ransomware"].value_counts())

Original training class distribution:
is_ransomware
0    2300227
1      33130
Name: count, dtype: int64


We can clearly see that there are much more `white ` transactions than `ransomware ` ones. If we were to adjust the training data so they would be equal, almost the entire dataset would be discarded. Instead, we will use augmentation (gausian augmentatioin) to artificialy increase the size of the `ransomware` set. We will add small values to already existing ones, that way, we will get new, similar, but not identical `ransomware` related transactions. The training set however will remain unchanged

In [10]:
# We create synthetic ransomware examples by copying ransomware rows
# and adding small random changes to their numerical feature values.

rng = np.random.default_rng(42)

desired_ransomware_fraction = 0.25

target_ransomware_count = int(
    desired_ransomware_fraction / (1 - desired_ransomware_fraction) * len(normal_train)
)

print(f"Target ransomware count: {target_ransomware_count:,}")

needed_synthetic_count = target_ransomware_count - len(ransomware_train)

ransomware_features = ransomware_train[feature_columns]

synthetic_ransomware = ransomware_features.sample(
    n=needed_synthetic_count,
    replace=True,
    random_state=42
).reset_index(drop=True)

# Add controlled noise to each feature.
# The noise is proportional to the feature's standard deviation.
noise_scale = 0.05

feature_std = ransomware_features.std().replace(0, 1)

noise = rng.normal(
    loc=0,
    scale=feature_std.values * noise_scale,
    size=synthetic_ransomware.shape
)

synthetic_ransomware = synthetic_ransomware + noise

# Some columns represent counts, so they should stay non-negative integers.
integer_like_columns = ["year", "day", "length", "count", "looped", "neighbors"]

for column in integer_like_columns:
    synthetic_ransomware[column] = synthetic_ransomware[column].round()

# Clip values so the synthetic records stay valid.
synthetic_ransomware["year"] = synthetic_ransomware["year"].clip(2009, 2018)
synthetic_ransomware["day"] = synthetic_ransomware["day"].clip(1, 365)

for column in feature_columns:
    synthetic_ransomware[column] = synthetic_ransomware[column].clip(lower=0)

synthetic_ransomware["is_ransomware"] = 1

synthetic_ransomware.head()

Target ransomware count: 766,742


,year,day,length,weight,count,looped,neighbors,income,is_ransomware
0,2015.0,172.0,4.0,0.388349,0.0,0.0,2.0,0.000000e+00,1
1,2013.0,277.0,9.0,0.506666,15.0,33.0,2.0,0.000000e+00,1
2,2017.0,109.0,9.0,0.117656,0.0,0.0,2.0,0.000000e+00,1
3,2014.0,78.0,2.0,0.553754,30.0,11.0,2.0,1.143587e+08,1
4,2013.0,300.0,6.0,1.916062,0.0,0.0,4.0,6.872918e+08,1


In [11]:
# Combine original normal rows, original ransomware rows, and synthetic ransomware rows.
augmented_train_df = pd.concat(
    [normal_train, ransomware_train, synthetic_ransomware],
    axis=0
).sample(frac=1, random_state=42)

X_train_augmented = augmented_train_df[feature_columns]
y_train_augmented = augmented_train_df["is_ransomware"]

print("Augmented training class distribution:")
print(y_train_augmented.value_counts())

print("\nAugmented training class percentages:")
print((y_train_augmented.value_counts(normalize=True) * 100).round(2))

Augmented training class distribution:
is_ransomware
0    2300227
1     766742
Name: count, dtype: int64

Augmented training class percentages:
is_ransomware
0    75.0
1    25.0
Name: proportion, dtype: float64


## 8. Scaling the Data
Next, we need to scale our splited dataset. Neural networks work better with datasets that are scaled, for example if one column is inside the interval (45,90) and all others are in the interval (1,10), than the first collumn will have a much larger inpact than the rest. So, all all colums have to use the same scale. To acomplish this, we will use `StandardScaler` which puts the columns on a comparable scale

In [12]:
scaler = StandardScaler()

# The train and test sets will be scaled separatelly to avoid leaking information from the training set to the testing set
X_train_scaled = scaler.fit_transform(X_train_augmented)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling complete.")

Feature scaling complete.


## 9. Baseline model
We will now train a baseline model that only predicts the most basic. This is not striclty necessary, but since the dataset that we use is imbalanced, having a baseline is highly usefull.

In [13]:
# We will try to predict the most common case, for usm that us the white adress, the non-ransomware case.
baseline_model = DummyClassifier(strategy="most_frequent")

# We will now train the model
baseline_model.fit(X_train_scaled, y_train_augmented)

#Next, we will predict the most common case
baseline_predictions = baseline_model.predict(X_test_scaled)

# Evaluating the baseline
baseline_accuracy = accuracy_score(y_test, baseline_predictions)
baseline_precision = precision_score(y_test, baseline_predictions, zero_division=0)
baseline_recall = recall_score(y_test, baseline_predictions, zero_division=0)
baseline_f1 = f1_score(y_test, baseline_predictions, zero_division=0)

print("Baseline Model Results")
print("----------------------")
print(f"Accuracy:  {baseline_accuracy:.4f}")
print(f"Precision: {baseline_precision:.4f}")
print(f"Recall:    {baseline_recall:.4f}")
print(f"F1 Score:  {baseline_f1:.4f}")

Baseline Model Results
----------------------
Accuracy:  0.9858
Precision: 0.0000
Recall:    0.0000
F1 Score:  0.0000


**Interpretation of the baseline results:**

`Accuracy:  0.9858` 

This is expected because the dataset is made up mostly of non-ransomware addresses. Since the baseline always predicts the most common class, it is correct most of the time.

`Precision: 0.0000` 

`Recall:    0.0000`

`F1 Score:  0.0000`

The other metrics are zero because the baseline does not identify any ransomware-related addresses. This shows that accuracy by itself is not a good enough metric for this problem, since it is heavily biased toward the majority class: `white`.

## 10. Multy Layer Perceptron Model
Now it's time to train the hole multy layer perceptron classifier. Unlike the baseline, this time, we will use the entire training data and uses the transaction graph features to learn patterns that may separate the normal Bitcoin adresses from the ransomware.

A multilayer perceptron is a neural network made of connected layers of artificial neurons. Each neuron receives input values, multiplies them by learned weights, adds a bias value, and passes the result through an activation function. During training, the model adjusts these weights so that its predictions become closer to the correct labels.

In this project, the input layer receives the 8 selected transaction graph features. The model then uses two hidden layers: one with 32 neurons and one with 16 neurons. The final output layer predicts whether an address is normal (`0`) or ransomware-related (`1`).

The `relu` activation function allows the model to learn non-linear patterns by keeping positive values and turning negative values into zero. 

The `adam` solver is the optimization algorithm used to update the network weights during training. It is commonly used because it adapts the learning rate while training.

`relu` was chosen because it is a strong default activation function for hidden layers in neural networks. It is simple, fast to compute, and helps the model learn non-linear relationships between the transaction graph features. This matters here because ransomware behavior is unlikely to be separated from normal behavior by one simple straight-line rule.

`adam` was chosen because it is a reliable general-purpose optimizer. It adapts the learning rate while training, which makes it easier to train a neural network without manually tuning the learning rate too much. This is useful for this project because the dataset is large, the features have different scales, and the goal is to get a working model in limited time.

The network uses two hidden layers with 32 and 16 neurons as a compromise between simplicity and flexibility. A very small model might not learn enough patterns, while a very large model would take longer to train and could overfit. The second layer is smaller than the first, which gradually compresses the learned information before the final binary prediction.



In [ ]:
# Definind model parameters
mlp_model = MLPClassifier(
    hidden_layer_sizes=(32, 16), # 2 hidden layers, one with 32 neurons one with 16
    activation="relu", # activation function 
    solver="adam",
    max_iter=30,
    random_state=42,
    early_stopping=False,
    verbose=True
)

# Train the model on the scaled training data.
mlp_model.fit(X_train_scaled, y_train_augmented)

mlp_predictions = mlp_model.predict(X_test_scaled)

mlp_accuracy = accuracy_score(y_test, mlp_predictions)
mlp_balanced_accuracy = balanced_accuracy_score(y_test, mlp_predictions)
mlp_precision = precision_score(y_test, mlp_predictions, zero_division=0)
mlp_recall = recall_score(y_test, mlp_predictions, zero_division=0)
mlp_f1 = f1_score(y_test, mlp_predictions, zero_division=0)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    mlp_predictions,
    labels=[0, 1]
).ravel()

print("MLP Model Results")
print("-----------------")
print(f"Iterations used:       {mlp_model.max_iter}")
print(f"Actual iterations:     {mlp_model.n_iter_}")
print(f"Final training loss:   {mlp_model.loss_:.4f}")
print()
print(f"Accuracy:              {mlp_accuracy:.4f}")
print(f"Balanced accuracy:     {mlp_balanced_accuracy:.4f}")
print(f"Precision:             {mlp_precision:.4f}")
print(f"Recall:                {mlp_recall:.4f}")
print(f"F1 Score:              {mlp_f1:.4f}")
print()
print(f"True negatives:        {tn}")
print(f"False positives:       {fp}")
print(f"False negatives:       {fn}")
print(f"True positives:        {tp}")

Iteration 1, loss = 0.33187696
Iteration 2, loss = 0.26206926
Iteration 3, loss = 0.25161041
Iteration 4, loss = 0.24614031
Iteration 5, loss = 0.24233134
Iteration 6, loss = 0.23921839
